# Relay games

A source pair plays the opening of a game; at relay point K a new pair inherits
the conversation and finishes it. With B a stronger model and A a weaker one:

```
B A B A | A A A A ...    K = 4
A A A A   A A A A ...    K = 0, the baseline: the weak pair on the same maze
```

Two steps: (a) build the base set of source games, (b) compare the relays
against the baseline.

In [ ]:
from notebook_helpers import set_working_dir_to_root
set_working_dir_to_root()

import pandas as pd
from src.utils.experiment_loading import get_experiment_data, add_derived_columns

## (a) Build the base set

`run_maze_task` draws source games from an overview CSV. Eight (o3 with
gpt-4.1-mini) are shipped; add your own collaborative job folders to use other
models.

In [ ]:
df_source = get_experiment_data(['example_relay_source'], add_messages=False, return_as_df=True)
add_derived_columns(df_source)
print(f'{len(df_source)} games')

OVERVIEW_FILE = './data/task_results/example_relay_source/relay_overview.csv'
OVERVIEW_COLUMNS = [
    'filepath', 'lm_1_name', 'lm_2_name', 'num_messages', 'outcome_success',
    'weighted_outcome', 'maze_size', 'split_1_proportion', 'split_2_proportion',
]
df_source[OVERVIEW_COLUMNS].sort_values('filepath').to_csv(OVERVIEW_FILE, index=False)
df_source[OVERVIEW_COLUMNS[1:]]

## (b) Compare the relays against the baseline

```bash
python -m src.create_maze_jobs --config_file experiments/job_configs/example_relay.yaml
bash experiments/jobs/example_relay/run_all_jobs.sh
```

In [ ]:
df = get_experiment_data(['example_relay'], add_messages=False, return_as_df=True)
add_derived_columns(df)
print(f'{len(df)} games')

by_k = df.groupby('pre_task_relay_point')['weighted_outcome'].agg(['mean', 'sem', 'size'])
by_k

In [ ]:
ax = by_k['mean'].plot(marker='o', yerr=by_k['sem'], capsize=4, figsize=(6, 4),
                       xlabel='relay point K', ylabel='weighted outcome')
ax.axhline(by_k['mean'].get(0, float('nan')), ls='--', c='grey', label='K = 0 baseline')
ax.legend();